In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv(
    "../data/processed/model_dataset.csv",
    index_col=0,
    parse_dates=True
)

print(df.shape)
df.head()

(2454, 18)


,Target_GK_Vol,Target_Vol_t1,AAPL_Return,Ret_Lag1,Ret_Lag5,Ret_Lag10,Vol_Roll5,Vol_Roll20,Vol_Roll60,Vol_Change,VIX_Close,VIX_Change,SPY_Return,SPY_Vol,MSFT_Return,MSFT_Vol,AAPL_Volume,Volume_Change
Date,,,,,,,,,,,,,,,,,,
2016-03-30,1.054690,0.512436,1.730836,2.339564,0.761877,1.989449,0.867093,0.975008,1.343805,-0.025540,13.56,-1.899248,0.437818,0.430684,0.619527,0.936978,182404400,0.379831
2016-03-31,0.512436,0.942546,-0.521597,1.730836,-0.554369,1.320394,0.819679,0.957179,1.325470,-0.018286,13.95,2.835518,-0.243004,0.351740,0.326437,0.879715,103553600,-0.566137
2016-04-01,0.942546,1.155689,0.913326,-0.521597,-0.434400,-0.160564,0.827529,0.970517,1.316621,0.013935,13.10,-6.286724,0.678862,0.760426,0.613720,1.200247,103496000,-0.000556
2016-04-04,1.155689,0.824183,1.022134,0.913326,-0.455278,0.113359,0.941198,0.948571,1.306780,-0.022612,14.12,7.497996,-0.324290,0.364468,-0.252242,0.843477,149424800,0.367260
2016-04-05,0.824183,1.067183,-1.185913,1.022134,2.339564,-0.009415,0.897909,0.926853,1.282988,-0.022896,15.42,8.807315,-1.003826,0.804057,-1.582015,0.813800,106314800,-0.340389


In [2]:
train_window = 1250

oos_df = df.iloc[train_window:].copy()

print(
    f"Out-of-Sample Observations: {len(oos_df)}"
)

Out-of-Sample Observations: 1204


In [3]:
oos_df["Forecast_Naive"] = (
    oos_df["Target_GK_Vol"]
)

In [4]:
oos_df["Forecast_HistAvg"] = (
    oos_df["Vol_Roll20"]
)

In [5]:
def calculate_metrics(
    actual,
    predicted
):

    epsilon = 1e-10

    actual = np.asarray(actual)
    predicted = np.asarray(predicted)

    mse = np.mean(
        (actual - predicted) ** 2
    )

    rmse = np.sqrt(mse)

    mae = np.mean(
        np.abs(actual - predicted)
    )

    actual_var = np.maximum(
        actual**2,
        epsilon
    )

    predicted_var = np.maximum(
        predicted**2,
        epsilon
    )

    qlike = np.mean(
        actual_var / predicted_var
        - np.log(
            actual_var /
            predicted_var
        )
        - 1
    )

    return {
        "QLIKE": qlike,
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae
    }

In [6]:
naive_metrics = calculate_metrics(
    oos_df["Target_Vol_t1"],
    oos_df["Forecast_Naive"]
)

naive_metrics

{'QLIKE': np.float64(0.44511059044943857),
 'MSE': np.float64(0.35961749793804193),
 'RMSE': np.float64(0.5996811635678095),
 'MAE': np.float64(0.43633364500032695)}

In [7]:
hist_metrics = calculate_metrics(
    oos_df["Target_Vol_t1"],
    oos_df["Forecast_HistAvg"]
)

hist_metrics

{'QLIKE': np.float64(0.3384618810886954),
 'MSE': np.float64(0.33860889044980824),
 'RMSE': np.float64(0.5819011002307938),
 'MAE': np.float64(0.39271606120647384)}

In [8]:
results = []

results.append({
    "Model":"Naive Forecast",
    **naive_metrics
})

results.append({
    "Model":"Historical Average",
    **hist_metrics
})

baseline_results = pd.DataFrame(
    results
)

baseline_results

,Model,QLIKE,MSE,RMSE,MAE
0,Naive Forecast,0.445111,0.359617,0.599681,0.436334
1,Historical Average,0.338462,0.338609,0.581901,0.392716


In [9]:
baseline_results.to_csv(
    "../reports/baseline_results.csv",
    index=False
)

In [10]:
baseline_results.sort_values(
    by="QLIKE"
)

,Model,QLIKE,MSE,RMSE,MAE
1,Historical Average,0.338462,0.338609,0.581901,0.392716
0,Naive Forecast,0.445111,0.359617,0.599681,0.436334
